# Control vs SDV: latent-variable causal models
Case 2 and Case 4 applied to paired CONN 19-ROI BOLD time series.

## 1. Setup

In [ ]:
from pathlib import Path
import json, os, sys, numpy as np, pandas as pd
from IPython.display import Image, display
ROOT=Path.cwd().parent if Path.cwd().name=='notebooks' else Path.cwd()
sys.path.insert(0,str(ROOT/'src'))
from causal_opt.fmri import *
from causal_opt.methods.static_latent import fit_static_latent
from causal_opt.methods.dynamic_latent import fit_dynamic_latent
DATA=ROOT.parent/'fmri_connectivity'/'data'/'mat_files'/'rs_sessions_r03_healthy'
SESSION1=Path(os.getenv('FMRI_SESSION1_ZIP',DATA/'roi_rs_sessions_Session1.zip'))
SESSION2=Path(os.getenv('FMRI_SESSION2_ZIP',DATA/'roi_rs_sessions_Session2.zip'))
OUTPUT=ROOT/'results'/'fmri_notebook'; OUTPUT.mkdir(parents=True,exist_ok=True)
INFERENCE_OUTPUT=Path(os.getenv('FMRI_INFERENCE_OUTPUT',ROOT/'results'/'fmri_case2_inference'))
SEED=1; K=2; P=1; EDGE_THRESHOLD=.3
STATIC=dict(k=K,lambda_w=.1,lambda_latent=.1,w_threshold=EDGE_THRESHOLD,random_state=SEED)
DYNAMIC=dict(k=K,lambda_0=.1,lambda_lag=.05,lambda_latent=.02,w0_threshold=EDGE_THRESHOLD,wlag_threshold=EDGE_THRESHOLD,random_state=SEED)
DISPLAY=tuple(f'BN19-{i:02d}' for i in range(1,20))

## 2. Load 19-ROI fMRI data

In [ ]:
control,sdv=load_fmri_state_data(SESSION1,SESSION2)
labels=SESSION1.parent/'labels'/'Bladder Network 19_labels.mat'
DISPLAY=load_roi_display_names(labels) if labels.exists() else DISPLAY
print(data_summary(control,'control / empty bladder')); print(data_summary(sdv,'full bladder / SDV'))
X_control=build_static_fmri_matrix(control); X_sdv=build_static_fmri_matrix(sdv)

## 3. Case 2 — static latent-confounded causal model

In [ ]:
static_control=fit_static_latent(X_control,**STATIC)
static_sdv=fit_static_latent(X_sdv,**STATIC)
print(static_control.diagnostics); print(static_sdv.diagnostics)

## 4. Case 2 — Control vs SDV comparison

In [ ]:
S_control=latent_similarity(static_control); S_sdv=latent_similarity(static_sdv)
plot_matrix_comparison(static_control.W0,static_sdv.W0,DISPLAY,('Control W','SDV W','SDV - Control'),OUTPUT/'static_W_comparison.png')
plot_matrix_comparison(S_control,S_sdv,DISPLAY,('Control LL^T','SDV LL^T','SDV - Control'),OUTPUT/'static_latent_comparison.png',True)
plot_graph_comparison(static_control.W0,static_sdv.W0,DISPLAY,OUTPUT/'static_graph_comparison.png',EDGE_THRESHOLD)
for f in ('static_W_comparison.png','static_latent_comparison.png','static_graph_comparison.png'): display(Image(filename=str(OUTPUT/f)))

## 5. Case 2 — Baseline comparison
Loads saved NOTEARS and endpoint-coded FCI results; it performs no fitting.

In [ ]:
baseline_csv=INFERENCE_OUTPUT/'case2_baseline_summary.csv'
if baseline_csv.exists():
    config_path=INFERENCE_OUTPUT/'case2_baseline_config.json'
    if config_path.exists(): display(json.loads(config_path.read_text()))
    display(pd.read_csv(baseline_csv))
    for f in ('static_control_proposed_vs_notears.png','static_sdv_proposed_vs_notears.png','notears_W_comparison.png','static_control_proposed_vs_notears_graph.png','static_sdv_proposed_vs_notears_graph.png','fci_pag_comparison.png','fci_skeleton_comparison.png'):
        if (INFERENCE_OUTPUT/f).exists(): display(Image(filename=str(INFERENCE_OUTPUT/f)))
else: print('Cached baseline results not found; no fitting is launched.')

## 6. Case 2 — Patient-wise bootstrap inference
This section reads saved full-data fits and paired-subject checkpoints only. Inference uses raw, pre-threshold W matrices. The current 95% CI is a percentile-bootstrap interval; `p_boot` is calculated separately from the centered bootstrap distribution. Therefore `CI excludes zero` and `p_boot < 0.05` are related but are **not guaranteed to be numerically equivalent**. Nominal classification uses `p_boot`, never CI exclusion or plotting-threshold appearance.

In [ ]:
summary_path=INFERENCE_OUTPUT/'case2_bootstrap_summary.json'; edge_csv=INFERENCE_OUTPUT/'case2_bootstrap_edge_results.csv'
if summary_path.exists() and edge_csv.exists():
    cached=json.loads(summary_path.read_text()); edges=pd.read_csv(edge_csv)
    nominal=edges[edges.p_boot<.05].sort_values(['p_boot','abs_delta_W'],ascending=[True,False])
    fdr=edges[edges.q_fdr<.05].sort_values(['q_fdr','abs_delta_W'],ascending=[True,False])
    print(f"Requested/completed/valid/failed: {cached['requested']}/{cached['completed']}/{cached['valid']}/{cached['failed']}")
    print(f"Valid paired bootstrap replicates: {cached['valid']}")
    print(f"Nominal p < 0.05 edges: {len(nominal)}")
    print(f"FDR q < 0.05 edges: {len(fdr)}")
    cols=['source_name','target_name','W_control_observed','W_sdv_observed','delta_W_observed','ci_2.5','ci_97.5','p_boot','q_fdr','ci_excludes_zero','control_selection_probability','sdv_selection_probability']
    print('Case 2 Control-to-SDV edges nominally significant (uncorrected p < 0.05)'); display(nominal[cols])
    print('FDR-supported edges (q < 0.05)'); display(fdr[cols] if len(fdr) else pd.DataFrame({'result':['No edge survives BH-FDR at q < 0.05.']}))
    focus=cached['left_insula_to_pag1']; display(pd.DataFrame([focus])); print(focus['nominal_statement']); print(focus['fdr_statement'])
    print('Five largest absolute observed state changes with final p/q values'); display(pd.DataFrame(cached['largest_absolute_changes']))
    display(pd.read_csv(INFERENCE_OUTPUT/'case2_node_reorganization.csv').head(12))
    for f in ('case2_bootstrap_delta_significance.png','case2_bootstrap_edge_stability.png','case2_bootstrap_nominal_edge_intervals.png','case2_bootstrap_key_edge_distributions.png','case2_node_reorganization.png'):
        if (INFERENCE_OUTPUT/f).exists(): display(Image(filename=str(INFERENCE_OUTPUT/f)))
else: print('Run scripts/summarize_fmri_case2_bootstrap.py on the completed cache; this cell does not fit models.')

## 7. Case 4 — time-series latent-confounded causal model

In [ ]:
X0c,Xlc,_=build_multisubject_lagged_data(control,P); X0s,Xls,_=build_multisubject_lagged_data(sdv,P)
dynamic_control=fit_dynamic_latent([control.timeseries[s] for s in control.subject_ids],P,**DYNAMIC)
dynamic_sdv=fit_dynamic_latent([sdv.timeseries[s] for s in sdv.subject_ids],P,**DYNAMIC)

## 8. Case 4 — Control vs SDV comparison

In [ ]:
plot_matrix_comparison(dynamic_control.W0,dynamic_sdv.W0,DISPLAY,('Control W0','SDV W0','SDV - Control W0'),OUTPUT/'dynamic_W0_comparison.png')
display(Image(filename=str(OUTPUT/'dynamic_W0_comparison.png')))
for tau in range(P):
    f=OUTPUT/f'dynamic_Wlag{tau+1}_comparison.png'; plot_matrix_comparison(dynamic_control.W_lags[tau],dynamic_sdv.W_lags[tau],DISPLAY,(f'Control W{tau+1}',f'SDV W{tau+1}',f'SDV - Control W{tau+1}'),f); display(Image(filename=str(f)))

## 9. Initial observations
Descriptive outputs only; no inferential or biological claims.